**BIO**

In [ ]:
import pandas as pd
import re

annotation_cols = [
    "Actions","Ingredient","Quantity","Unit","State","Form",
    "Dry/Fresh","Size","PreProcessing","Time","Temperature","Utensil"
]

def extract_temp(text):
    if pd.isna(text):
        return text
    text = str(text)
    match = re.search(r'(\d+)\s*[°]*\s*[fFcC]?', text)
    if match:
        return match.group(1)
    return text


def bio_encode(row):
    instruction_tokens = [
        extract_temp(tok) for tok in str(row["Instruction"]).replace('.', ' ').replace(',', ' ').split()
    ]
    tags = ["O"] * len(instruction_tokens)

    for col in annotation_cols:
        if pd.isna(row[col]):
            continue

        values = str(row[col]).split("|")

        for val in values:
            val = extract_temp(val) if col == "Temperature" else str(val)
            val_tokens = val.split()
            n = len(val_tokens)

            for i in range(len(instruction_tokens) - n + 1):
                if instruction_tokens[i:i+n] == val_tokens:
                    tags[i] = f"B-{col}"
                    for j in range(1, n):
                        tags[i+j] = f"I-{col}"

    return " ".join(tags)


df = pd.read_csv("/content/RecipeDB3_Instruction_Annotation - Sheet1.csv")
df["BIO"] = df.apply(bio_encode, axis=1)

output_path = "RecipeDB3_BIO.csv"
df.to_csv(output_path, index=False)

**Spacy**

In [ ]:
!pip install spacy seqeval scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1c7de05ca3ecf8626e090979d056cbfac6a36445f24267f090d6a56cfd4682aa
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report

COLUMN_LABEL_MAP = {
    "Actions": "ACTION",
    "Ingredient": "INGREDIENT",
    "Quantity": "QUANTITY",
    "Unit": "UNIT",
    "State": "STATE",
    "Form": "FORM",
    "Dry/Fresh": "DRYFRESH",
    "Size": "SIZE",
    "PreProcessing": "PREPROCESS",
    "Time": "TIME",
    "Temperature": "TEMPERATURE",
    "Utensil": "UTENSIL",
}

df = pd.read_csv('/content/RecipeDB3_BIO.csv')
df['Instruction'] = df['Instruction'].fillna('').astype(str)
df['BIO'] = df['BIO'].fillna('').astype(str)

def bio_to_spacy_format(sentence, tags, column_label_map):
    tokens = sentence.split()
    bio_tags = tags.split()
    if len(tokens) != len(bio_tags):
        return None

    offsets = []
    start = 0
    for token in tokens:
        start = sentence.find(token, start)
        end = start + len(token)
        offsets.append((start, end))
        start = end

    entities = []
    chunk_start = None
    chunk_end = None
    chunk_label = None
    for i, tag in enumerate(bio_tags):
        if tag == "O":
            if chunk_start is not None and chunk_label is not None:
                entities.append((chunk_start, chunk_end, chunk_label))
                chunk_start, chunk_end, chunk_label = None, None, None
        else:
            scheme, value = tag.split('-', 1)
            mapped_label = None
            for short, full in column_label_map.items():
                if short.lower() == value.lower() or full.lower() == value.lower():
                    mapped_label = full
            if scheme == "B":
                if chunk_start is not None and chunk_label is not None:
                    entities.append((chunk_start, chunk_end, chunk_label))
                chunk_start = offsets[i][0]
                chunk_end = offsets[i][1]
                chunk_label = mapped_label
            elif scheme == "I" and chunk_label == mapped_label:
                chunk_end = offsets[i][1]
            else:
                if chunk_start is not None and chunk_label is not None:
                    entities.append((chunk_start, chunk_end, chunk_label))
                chunk_start = offsets[i][0]
                chunk_end = offsets[i][1]
                chunk_label = mapped_label
    if chunk_start is not None and chunk_label is not None:
        entities.append((chunk_start, chunk_end, chunk_label))
    return (sentence, {"entities": entities})

TRAIN_DATA = [
    x for x in
    (bio_to_spacy_format(row['Instruction'], row['BIO'], COLUMN_LABEL_MAP) for _, row in df.iterrows())
    if x is not None
]

train_data, test_data = train_test_split(TRAIN_DATA, test_size=0.2, random_state=42)

nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

for label in COLUMN_LABEL_MAP.values():
    ner.add_label(label)

optimizer = nlp.begin_training()
for itn in range(10):
    losses = {}
    batches = minibatch(train_data, size=8)
    for batch in batches:
        examples = []
        for text, annotations in batch:
            examples.append(Example.from_dict(nlp.make_doc(text), annotations))
        nlp.update(examples, drop=0.3, losses=losses)
    print(f"Iteration {itn}: Losses {losses}")

nlp.to_disk('./spacy-ner-cooking-final')

def align_entities(text, labels, nlp_model):
    doc = nlp_model(text)
    tokens = [token.text for token in doc]

    true_ents = ['O'] * len(tokens)
    pred_ents = ['O'] * len(tokens)

    char_to_token = {}
    for i, token in enumerate(doc):
        for pos in range(token.idx, token.idx + len(token.text)):
            char_to_token[pos] = i

    for start, end, label in labels["entities"]:
        if start in char_to_token and (end - 1) in char_to_token:
            start_idx = char_to_token[start]
            end_idx = char_to_token[end - 1]
            true_ents[start_idx] = f"B-{label}"
            for i in range(start_idx + 1, end_idx + 1):
                true_ents[i] = f"I-{label}"

    for ent in doc.ents:
        if ent.start_char in char_to_token and (ent.end_char - 1) in char_to_token:
            start_idx = char_to_token[ent.start_char]
            end_idx = char_to_token[ent.end_char - 1]
            pred_ents[start_idx] = f"B-{ent.label_}"
            for i in range(start_idx + 1, end_idx + 1):
                pred_ents[i] = f"I-{ent.label_}"

    return true_ents, pred_ents

y_true, y_pred = [], []
for text, ann in test_data:
    t, p = align_entities(text, ann, nlp)
    y_true.append(t)
    y_pred.append(p)

results = classification_report(y_true, y_pred, output_dict=True)
print(f"F1 Score: {results['micro avg']['f1-score']:.4f}")
for entity, stats in results.items():
    if entity not in ('O', 'micro avg', 'macro avg'):
        print(f"Entity {entity}: F1 = {stats['f1-score']:.4f} Precision = {stats['precision']:.4f} Recall = {stats['recall']:.4f}")


Iteration 0: Losses {'ner': np.float32(17442.246)}
Iteration 1: Losses {'ner': np.float32(8143.8735)}
Iteration 2: Losses {'ner': np.float32(6724.486)}
Iteration 3: Losses {'ner': np.float32(6045.273)}
Iteration 4: Losses {'ner': np.float32(5652.786)}
Iteration 5: Losses {'ner': np.float32(5406.3965)}
Iteration 6: Losses {'ner': np.float32(5125.6953)}
Iteration 7: Losses {'ner': np.float32(4925.3955)}
Iteration 8: Losses {'ner': np.float32(4839.814)}
Iteration 9: Losses {'ner': np.float32(4671.6133)}
F1 Score: 0.9258
Entity ACTION: F1 = 0.9039 Precision = 0.9058 Recall = 0.9021
Entity DRYFRESH: F1 = 0.8308 Precision = 0.7500 Recall = 0.9310
Entity FORM: F1 = 0.9787 Precision = 0.9705 Recall = 0.9871
Entity INGREDIENT: F1 = 0.9684 Precision = 0.9581 Recall = 0.9789
Entity PREPROCESS: F1 = 0.5000 Precision = 0.8571 Recall = 0.3529
Entity QUANTITY: F1 = 0.6381 Precision = 0.6262 Recall = 0.6505
Entity SIZE: F1 = 0.9239 Precision = 0.9412 Recall = 0.9072
Entity STATE: F1 = 0.9844 Precision

In [ ]:
import pandas as pd
import spacy

nlp = spacy.load("./spacy-ner-cooking-final")

ENTITY_COLUMNS = [
    "Actions", "Ingredient", "Quantity", "Unit", "State", "Form",
    "Dry/Fresh", "Size", "PreProcessing", "Time", "Temperature", "Utensil"
]

COLUMN_LABEL_MAP = {
    "Actions": "ACTION",
    "Ingredient": "INGREDIENT",
    "Quantity": "QUANTITY",
    "Unit": "UNIT",
    "State": "STATE",
    "Form": "FORM",
    "Dry/Fresh": "DRYFRESH",
    "Size": "SIZE",
    "PreProcessing": "PREPROCESS",
    "Time": "TIME",
    "Temperature": "TEMPERATURE",
    "Utensil": "UTENSIL",
}

input_csv_path = "/content/RecipeDB3_Instruction_Annotation - Sheet1.csv"
df = pd.read_csv(input_csv_path)
df['Instruction'] = df['Instruction'].fillna('').astype(str)

for col in ENTITY_COLUMNS:
    df[col] = ""

def extract_entities_spacy(text, nlp_model, column_label_map, entity_columns):
    result = {col: [] for col in entity_columns}
    doc = nlp_model(text)
    inv_label_map = {v: k for k, v in column_label_map.items()}
    for ent in doc.ents:
        col = inv_label_map.get(ent.label_)
        if col:
            result[col].append(ent.text)
    return {col: ", ".join(spans) if spans else "" for col, spans in result.items()}

for i, row in df.iterrows():
    instr = str(row['Instruction'])
    entities = extract_entities_spacy(instr, nlp, COLUMN_LABEL_MAP, ENTITY_COLUMNS)
    for col in ENTITY_COLUMNS:
        df.at[i, col] = entities[col]

output_columns = [
    "Recipe_ID", "Recipe Name", "Instruction"
] + ENTITY_COLUMNS
output_path = "/content/spacy_results.csv"
df.to_csv(output_path, columns=output_columns, index=False)

print(f"Saved extracted recipe NER output to {output_path}")


Saved extracted recipe NER output to /content/spacy_results.csv


In [ ]:
import shutil

model_dir = "./spacy-ner-cooking-final"
zip_path = "./spacy-ner-cooking-final.zip"

shutil.make_archive(base_name=zip_path.replace('.zip', ''), format='zip', root_dir=model_dir)

print(f"Zipped model saved as {zip_path}")


Zipped model saved as ./spacy-ner-cooking-final.zip


In [ ]:
import pandas as pd

rows = []

print(f"{'Entity':>14} {'Precision':>9} {'Recall':>9} {'F1-Score':>9} {'Support':>9}")


for entity, stats in results.items():
    if entity in ('O', 'micro avg', 'macro avg', 'weighted avg'):
        continue

    print(
        f"{entity:>14} {stats['precision']:>9.4f} {stats['recall']:>9.4f} "
        f"{stats['f1-score']:>9.4f} {int(stats['support']):>9}"
    )


for key in ['micro avg', 'macro avg', 'weighted avg']:
    if key in results:
        stats = results[key]
        print(
            f"{key:>14} {stats['precision']:>9.4f} {stats['recall']:>9.4f} "
            f"{stats['f1-score']:>9.4f} {int(stats['support']):>9}"
        )

        Entity Precision    Recall  F1-Score   Support
        ACTION    0.9058    0.9021    0.9039      2686
      DRYFRESH    0.7500    0.9310    0.8308        29
          FORM    0.9705    0.9871    0.9787       233
    INGREDIENT    0.9581    0.9789    0.9684      1471
    PREPROCESS    0.8571    0.3529    0.5000        17
      QUANTITY    0.6262    0.6505    0.6381       103
          SIZE    0.9412    0.9072    0.9239       194
         STATE    0.9731    0.9961    0.9844       254
   TEMPERATURE    0.8266    0.9404    0.8798       218
          TIME    0.9469    0.9469    0.9469       245
          UNIT    0.9441    0.9441    0.9441       143
       UTENSIL    0.9459    0.9309    0.9383       752
     micro avg    0.9227    0.9289    0.9258      6345
     macro avg    0.8871    0.8723    0.8698      6345
  weighted avg    0.9232    0.9289    0.9255      6345
